In [1]:
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sys.path.append("../src")

df = pd.read_pickle("../data/processed/itsm_clean.pkl")
print(df.shape)

(46606, 26)


In [2]:
raw_cols = pd.read_csv("../data/raw/itsm_raw.csv", nrows=0).columns
print("Added  :", sorted(set(df.columns) - set(raw_cols)))
print("Removed:", sorted(set(raw_cols) - set(df.columns)))

Added  : ['Elapsed_hrs', 'Has_Related_Change', 'Has_Reopen_Time']
Removed: ['Alert_Status', 'number_cnt']


## Target balance

In [3]:
lab = df[df["Priority"].notna()]
print(len(lab))
print(lab["Priority"].astype(int).value_counts().sort_index())
print("High (P1+P2) share:", round(lab["Priority"].isin([1, 2]).mean() * 100, 2), "%")

45226
Priority
1        3
2      697
3     5323
4    22717
5    16486
Name: count, dtype: int64
High (P1+P2) share: 1.55 %


## volume over time for Track 2

In [ ]:
monthly = df.set_index("Open_Time").resample("MS").size()
monthly.plot(figsize=(10, 3), title="Tickets opened per month")
plt.show()
print(monthly.describe().round(0))

In [ ]:
for c in ["CI_Cat", "Category", "Closure_Code"]:
    print(df[c].value_counts().head(8), "\n")

In [ ]:
cap = df["Elapsed_hrs"].quantile(0.99)
df["Elapsed_hrs"].clip(upper=cap).plot.hist(bins=50, figsize=(8, 3), title="Elapsed hours, capped at the 99th percentile")
plt.show()

In [ ]:
closed = df.set_index("Close_Time").resample("MS").size()
both = pd.DataFrame({"opened": monthly, "closed": closed}).fillna(0).astype(int)
print(both.loc["2013-07":])
print("Share opened Oct 2013 or later:", round((df["Open_Time"] >= "2013-10-01").mean() * 100, 1), "%")
print("Share closed Oct 2013 or later:", round((df["Close_Time"] >= "2013-10-01").mean() * 100, 1), "%")
print("Close_Time range:", df["Close_Time"].min().date(), "to", df["Close_Time"].max().date())

In [ ]:
tot = lab.set_index("Open_Time").resample("MS").size()
hi = lab[lab["Priority"].isin([1, 2])].set_index("Open_Time").resample("MS").size()
print(pd.DataFrame({"labelled": tot, "high": hi}).fillna(0).astype(int).loc["2013-09":])

1. The 1,150 tickets opened before October are the backlog still open on 1 Oct (Sep 857, Aug 93, Jul 73, about 127 earlier). They are in the file only because they were slow, which is survivorship bias. High-priority tickets get fixed fast, so they almost never appear there. By my arithmetic, that is 1 High in 1,114 labelled (0.09%), against 699 in 44,112 (1.58%) from October on. Cell 8 checks this. Any feature that reveals “opened before October” would let a model learn “old means low priority”, which is an artifact.
2. Track 2 has 182 days of data. Quarterly forecasting is 2 data points and annual is none, so the brief’s quarterly and annual forecasts can’t be done. Daily or weekly forecasting works, with about 26 weeks and no yearly seasonality. Report this limit rather than working around it.
3. The High rate drifts down. A threshold tuned on October will over-alert in March.
4. Test sets are small. With 79 High cases in one month, recall is uncertain by roughly ±11 points. With 158 cases it is about ±8.

1. Modelling window: Open_Time on or after 2013-10-01. That is 45,456 rows, 44,112 labelled, 699 High. Earlier tickets stay in EDA and in one sensitivity check.
2. Track 1 split: train and tune on Oct to Jan with an expanding window (TimeSeriesSplit). That is 30,845 labelled and 541 High. Use Feb to Mar as the final test (13,267 labelled, 158 High) and touch it once. This replaces my earlier suggestion of a March-only test.
3. Track 2: daily or weekly counts only,

## confirms the survivorship effect

In [ ]:
pre = lab[lab["Open_Time"] < "2013-10-01"]
win = lab[lab["Open_Time"] >= "2013-10-01"]
for name, part in [("Opened before Oct 2013", pre), ("Opened Oct 2013 onward", win)]:
    hi = part["Priority"].isin([1, 2]).sum()
    print(f"{name}: labelled {len(part)}, High {hi}, rate {hi / len(part) * 100:.2f}%")

## Tickets opened before Oct 2013 are rarely High priority


checks whether late March is cut off. Tickets opened in late March and still open on 31 March are missing from the file. Your opened-versus-closed drop in March (-14% against -4% in closed) hints at it but doesn’t prove it:

In [ ]:
w = df[df["Open_Time"] >= "2013-10-01"]
wk = w.groupby(w["Open_Time"].dt.to_period("W")).agg(
    opened=("Incident_ID", "size"),
    median_elapsed_hrs=("Elapsed_hrs", "median"),
)
print(wk.round(1).tail(8))

In [ ]:
print(df.loc[df["Status"] != "Closed", ["Open_Time", "Close_Time"]].agg(["min", "max"]))

## monthly chart: marked the window start and labelled the axes

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(monthly.index, monthly.values)
ax.axvline(pd.Timestamp("2013-10-01"), color="red", linestyle="--")
ax.set_title("Tickets opened per month (dashed: start of close-date window)")
ax.set_ylabel("Tickets")
plt.show()

## Diagnostic cell to run next (aggregates only)

How to read it:

1. If p90_hrs and share_over_72h stay flat for weeks and then drop, censoring is confirmed, and the first week of the drop sets the cut-off.
2. If High tickets have a much lower median elapsed time than the rest, the High-rate inflation is plausible.
3. The weekly High counts show the rate drifting in the last weeks.

In [ ]:
w = df[df["Open_Time"] >= "2013-10-01"].copy()
w["week"] = w["Open_Time"].dt.to_period("W")

tail = w.groupby("week").agg(
    opened=("Incident_ID", "size"),
    median_hrs=("Elapsed_hrs", "median"),
    p90_hrs=("Elapsed_hrs", lambda s: s.quantile(0.9)),
    share_over_72h=("Elapsed_hrs", lambda s: (s > 72).mean() * 100),
)
print(tail.round(1).tail(10))

lab_w = w[w["Priority"].notna()].copy()
lab_w["high"] = lab_w["Priority"].isin([1, 2])
print(lab_w.groupby("week")["high"].agg(["size", "sum"]).tail(10))
print(lab_w.groupby("high")["Elapsed_hrs"].median().round(1))